# Step 1: preprocessing

Raw recordings → 3 clean tables in `data/processed/`: **sessions**, **trials**, **frames**.

Preprocessing only **organises and flags** the data: no smoothing, no filtering, no change of coordinates. Nothing is deleted: doubtful sessions and trials get a flag (`is_usable`, `passes_checks`) and later steps decide what to use.

All the code is in `preprocess.py`. All paths and settings are in `config.py`.

In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd

import preprocess
from config import MAX_HEAD_HEIGHT_M, MAX_WRONG_PERSON_PCT, MIN_FRAMES_PER_TRIAL, MIN_TRIALS_ONE_TASK, RAW_ROOT

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

## The raw data

**session** = one recording (one visit can have several) · **task** = one of the 8 exercises · **trial** = one repetition of a task · **frame** = one snapshot of the 15 body points (16 to 43 per second, depending on the session).

One session = 3 files with the same name `upper_limb_<date>_<time>`:
- `.events.jsonl`: what happened and when (task start, trial start, target reached, trial end, ...)
- `.frames.csv`: one row per frame, 15 body points × (x, y, z, confidence)
- `.meta.json`: the settings of the session (missing for 4 sessions)

Calibration recordings (`r1_calibration_*`) are not part of this step.

In [2]:
# A session = an events file "upper_limb_*.events.jsonl". The raw files are only read, never modified.
session_ids = preprocess.list_sessions()
print(f"Raw folder: {RAW_ROOT}")
print(f"Sessions found: {len(session_ids)}")

Raw folder: /Users/louistschanz/Documents/EPFL/Champalimaud/immersive-kinematics/data/upper_limb_recorded_data/upper_limb_asessment/recordings
Sessions found: 112


## 1. Sessions table (one row per session)

For each session: **when** (date, start time), the **ID typed** by the operator, **how much data** (frames, trials, tasks, arms, minutes of tasks), the **setup_version** (see below) and **is_usable**.

Every session is in the table. `is_usable` marks the sessions that go on to the trials and frames tables:
- it has frames and a `meta.json` file,
- at least `MIN_TRIALS_ONE_TASK` repetitions of one task with one arm: quick recording tests have 1 to 5,
- the person sat: `head_height_m` (nose height during the repetitions) not above `MAX_HEAD_HEIGHT_M`. Seated persons are at ~1.1–1.3 m; ~1.5–1.9 m means staff standing to test the setup.

In [3]:
sessions = preprocess.build_sessions()

# Why sessions are not usable (a session can have several reasons).
not_usable = sessions[~sessions["is_usable"]]
print(f"Sessions: {len(sessions)}, usable: {sessions['is_usable'].sum()}, not usable: {len(not_usable)}")
print(f"  no frames:                                 {(not_usable['n_frames'] == 0).sum()}")
print(f"  no meta.json:                              {(~not_usable['has_meta']).sum()}")
print(f"  fewer than {MIN_TRIALS_ONE_TASK} repetitions of one task:     {(not_usable['most_trials_one_task'] < MIN_TRIALS_ONE_TASK).sum()}")
print(f"  person standing (head above {MAX_HEAD_HEIGHT_M} m):       {(not_usable['head_height_m'] > MAX_HEAD_HEIGHT_M).sum()}")

sessions[sessions["is_usable"]].head()

Sessions: 112, usable: 29, not usable: 83
  no frames:                                 6
  no meta.json:                              4
  fewer than 4 repetitions of one task:     71
  person standing (head above 1.42 m):       39


,session_id,date,start_time,typed_id,n_frames,n_trials,tasks,arms,task_time_min,head_height_m,most_trials_one_task,has_meta,setup_version,is_usable
0,upper_limb_20260629_090253,2026-06-29,09:02,None,453,10,1,right,0.3,1.23,10,True,51839393,True
1,upper_limb_20260629_090948,2026-06-29,09:09,None,829,5,6,right,0.4,1.22,5,True,51839393,True
2,upper_limb_20260629_091039,2026-06-29,09:10,None,1265,30,6,right,0.9,1.21,30,True,51839393,True
3,upper_limb_20260629_095022,2026-06-29,09:50,None,14424,75,"1, 2, 3, 4, 5, 6",right,12.5,1.29,30,True,51839393,True
4,upper_limb_20260629_100413,2026-06-29,10:04,None,3416,31,"6, 7",right,3.2,1.24,30,True,51839393,True


### Setup versions

The setup and the task settings changed between June and September. `setup_version` is a short fingerprint of the settings in `meta.json` that change how the tasks run or how the body is tracked (`SETUP_KEYS` in `config.py`): how a trial ends, the task thresholds and targets, the body tracking. Values that change every session (times, IDs, file paths) and the infrastructure (servers, audio, projector) are left out. Same settings = same fingerprint.

The table shows each version in order of first use, and only the settings that differ between versions (the task thresholds never changed). A setting missing from `meta.json` shows as `None`: it was not saved yet in the older recordings, which does not always mean the setup was different.

In [4]:
preprocess.setup_versions_table(sessions)

,first_date,last_date,n_sessions,n_usable,tasks,automatic_trial_end_initial,manual_trial_end,table_wrist_yaw_variant_task_ids,table_wrist_yaw_variants_deg,pose_frame_source_mode,pose_triangulation_profile,pose_y_correction_m
setup_version,,,,,,,,,,,,
51839393,2026-06-29,2026-07-06,21,15,"1, 2, 3, 4, 5, 6, 7, 8",None,None,None,None,None,None,None
afd22e92,2026-07-08,2026-07-08,4,0,1,None,False,None,None,None,None,None
53563a21,2026-07-08,2026-07-08,2,1,"1, 6",None,False,None,None,None,None,0.1
8000ba31,2026-07-08,2026-07-10,5,2,"1, 2",None,True,"[6, 7]","{'left': [0.0, -45.0], 'right': [0.0, 45.0]}",None,None,0.1
f71e1611,2026-07-08,2026-07-28,19,3,"1, 2, 3, 4, 5, 6, 7, 8",None,False,"[6, 7]","{'left': [0.0, -45.0], 'right': [0.0, 45.0]}",None,None,0.1
70971bd9,2026-07-28,2026-07-28,3,0,"1, 2",None,False,"[6, 7]","{'left': [0.0, -45.0], 'right': [0.0, 45.0]}",optitrack_live,None,0.1
faba0a58,2026-07-29,2026-08-11,28,3,"1, 2, 3, 4, 5, 6, 7, 8",None,False,"[6, 7]","{'left': [0.0, -45.0], 'right': [0.0, 45.0]}",optitrack_live,robust,0.1
9d0a86a7,2026-09-02,2026-09-24,26,5,"1, 2, 3, 4, 5, 6, 7, 8",False,False,"[6, 7]","{'left': [0.0, -45.0], 'right': [0.0, 45.0]}",optitrack_live,robust,0.1


### Participants

Who is who is written **by hand** in 2 CSV files in `data/` (not in the code):
- `session_participant.csv`: one row per session. `participant_id` is prefilled with the typed ID; fill in the sessions without one (and correct wrong typed IDs), and `visit` (1 = first assessment, 2 = reassessment).
- `participants.csv`: one row per participant: `condition` (`healthy`, the diagnosis, or `unknown`), the visit dates given by the clinical team (for reference), notes.

They are written once as templates, and **never overwritten** by the code: run this as often as you like. If a session is missing from `session_participant.csv` (a new recording), a warning lists it: add it by hand.

Careful with Excel: it removes the leading zeros of IDs like `004`.

In [5]:
preprocess.make_participant_templates(sessions)
sessions = preprocess.add_participant_id(sessions)

session_participant.csv: already exists, not changed
participants.csv: already exists, not changed


In [6]:
# Counts, for all sessions and for the usable ones. Tests = typed ID starting with "test" (test_33, tester, ...).
for name, table in [("all", sessions), ("usable", sessions[sessions["is_usable"]])]:
    is_test = table["typed_id"].str.lower().str.startswith("test", na=False)
    print(f"{name:<7} sessions: {len(table):>3}  |  typed ID: {table['typed_id'].notna().sum():>3}, "
          f"no typed ID: {table['typed_id'].isna().sum():>3}  |  participant_id filled in: "
          f"{table['participant_id'].notna().sum():>3}  |  tests: {is_test.sum():>3}")

all     sessions: 112  |  typed ID:  31, no typed ID:  81  |  participant_id filled in:  40  |  tests:  21
usable  sessions:  29  |  typed ID:   8, no typed ID:  21  |  participant_id filled in:  14  |  tests:   0


## 2. Trials table (one row per trial = one repetition)

Only for the usable sessions. A trial = everything between a `trial_start` event and the next `trial_end`. **Every trial is kept**, also repetition 1 and failed trials.

Good to know:
- `task_run`: the operator sometimes starts the same task again in a session (after a mistake, or for the other arm). The new run has `task_run = 2` and its `trial_index` starts again from 1.
- `is_first_repetition`: repetition 1 of each run includes a ~5 s countdown and is often a practice movement.
- `pose_locked_time`: the person is in the start position, the movement can begin.
- `goal_time`: time of the first goal event (target / circle / cup / mouth / nose reached), empty if none. `goal_events`: all of them (type + time). All times are the software clock (`wall_time`, seconds).
- `end_type`: **auto** = the person came back to the start · **manual** = the operator pressed a key · **failed** = came back without reaching the target · **interrupted** = recording stopped or task restarted. From 2 September the operator ends every trial by hand.

In [7]:
usable = sessions[sessions["is_usable"]]
trials = preprocess.build_trials(usable)
print(f"Trials: {len(trials)} in {trials['session_id'].nunique()} sessions")
trials.head()

Trials: 2803 in 29 sessions


,trial_id,session_id,task_id,task_name,task_run,trial_index,is_first_repetition,side,target,start_time,pose_locked_time,goal_time,end_time,duration_s,end_type,goal_reached,goal_events,n_frames
0,upper_limb_20260629_090253_001,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,1,True,right,90,1.782720e+09,1.782720e+09,1.782720e+09,1.782720e+09,7.013817,auto,True,"[{'type': 'target_reached', 'time': 1782720196...",123
1,upper_limb_20260629_090253_002,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,2,False,right,90,1.782720e+09,1.782720e+09,1.782720e+09,1.782720e+09,1.452592,auto,True,"[{'type': 'target_reached', 'time': 1782720198...",25
2,upper_limb_20260629_090253_003,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,3,False,right,90,1.782720e+09,1.782720e+09,1.782720e+09,1.782720e+09,1.393823,auto,True,"[{'type': 'target_reached', 'time': 1782720199...",24
3,upper_limb_20260629_090253_004,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,4,False,right,90,1.782720e+09,1.782720e+09,1.782720e+09,1.782720e+09,1.458956,auto,True,"[{'type': 'target_reached', 'time': 1782720201...",26
4,upper_limb_20260629_090253_005,upper_limb_20260629_090253,1,Shoulder flexion with elbow extended,1,5,False,right,90,1.782720e+09,1.782720e+09,1.782720e+09,1.782720e+09,1.388258,auto,True,"[{'type': 'target_reached', 'time': 1782720202...",24


In [8]:
# Per task: number of trials, which arm, and how the trials ended.
by_task = [trials["task_id"], trials["task_name"]]
summary = pd.concat([pd.crosstab(by_task, trials["side"], margins=True).drop(columns="All"),
                     pd.crosstab(by_task, trials["end_type"], margins=True).drop(columns="All")], axis=1)
summary.insert(0, "trials", summary["left"] + summary["right"])
summary

,,trials,left,right,auto,failed,interrupted,manual
task_id,task_name,,,,,,,
1,Shoulder flexion with elbow extended,307,121,186,162,0,2,143
2,Shoulder abduction with elbow extended,278,116,162,158,0,1,119
3,Draw a 5 segment trajectory,216,89,127,143,0,2,71
4,Reaching different targets,544,210,334,376,0,1,167
5,Draw a circle,215,84,131,143,0,1,71
6,Horizontal reaching,721,252,469,502,3,5,211
7,Drinking test: reaching glass to mouth,278,125,153,114,0,3,161
8,Index to nose,244,108,136,120,0,2,122
All,,2803,1105,1698,1718,3,17,1065


Which goal event each trial reached first (it gives `goal_time`), per task. `not reached` = no goal event in the trial.

These events come from the recording software (events file), not from our code: each task shows a goal (tasks 1 and 2: a circle at 90°, task 4: numbered targets, task 6: targets on the table, task 5: the circle, task 7: cup and mouth, task 8: nose), and the software writes an event when the hand gets there. Most tasks also write a general `target_reached` after their own goal event.

⚠️ Task 7 (drinking): `cup_reached` is often missing, so `goal_time` is sometimes the cup and sometimes the mouth. `goal_events` keeps all of them, so a later step can choose.

In [9]:
first_goal = trials["goal_events"].str[0].str.get("type").fillna("not reached")
goals = pd.crosstab([trials["task_id"], trials["task_name"]], first_goal)
goals.insert(0, "trials", goals.sum(axis=1))   # total number of trials of the task
goals

,goal_events,trials,circle_completed,cup_reached,mouth_reached,nose_reached,not reached,target_reached
task_id,task_name,,,,,,,
1,Shoulder flexion with elbow extended,307,0,0,0,0,2,305
2,Shoulder abduction with elbow extended,278,0,0,0,0,5,273
3,Draw a 5 segment trajectory,216,0,0,0,0,10,206
4,Reaching different targets,544,0,0,0,0,18,526
5,Draw a circle,215,201,0,0,0,14,0
6,Horizontal reaching,721,0,0,0,0,11,710
7,Drinking test: reaching glass to mouth,278,0,110,147,0,21,0
8,Index to nose,244,0,0,0,241,3,0


## 3. Frames table (one row per frame inside a trial)

Only the frames inside a trial are kept, each with its `trial_id`. Columns: `trial_id`, `time_s`, `is_main_person`, then x / y / z / conf of 14 body points (`neck` is left out: always empty). Positions in metres, room coordinates (y = height), as recorded.

Good to know:
- `time_s`: 0 = person in the start position. **< 0** = countdown / waiting (mostly repetition 1), **≥ 0** = movement. It uses the camera clock (`pose_t`), which is regular (the software clock is not).
- `is_main_person`: the camera sometimes follows another person (the operator). The person sits still during a task, so a frame is the main person if the shoulders are within 30 cm of their usual place in that task run. Only marked, not deleted.
- Missing points stay empty (NaN), never 0.

In [10]:
frames = preprocess.build_frames(trials)
print(f"Frames: {len(frames):,} rows x {frames.shape[1]} columns, in {frames['trial_id'].nunique()} trials")
print(f"Memory: {frames.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"Countdown / waiting frames (time_s < 0): {(frames['time_s'] < 0).mean():.1%}")
print(f"Frames of another person (is_main_person = False): {(~frames['is_main_person']).mean():.1%}")

Frames: 429,731 rows x 59 columns, in 2802 trials
Memory: 138 MB
Countdown / waiting frames (time_s < 0): 3.7%
Frames of another person (is_main_person = False): 1.9%


In [11]:
# % of frames where each value is missing, per body point.
missing = frames[preprocess.KEYPOINT_COLUMNS].isna().mean().mul(100).round(2)
missing.index = pd.MultiIndex.from_tuples([c.rsplit("_", 1) for c in missing.index], names=["keypoint", "value"])
missing.unstack().loc[preprocess.KEYPOINTS, ["x", "y", "z", "conf"]]

value,x,y,z,conf
keypoint,,,,
nose,0.12,0.12,0.12,0.12
left_eye,0.22,0.22,0.22,0.22
right_eye,0.33,0.33,0.33,0.33
left_ear,0.28,0.28,0.28,0.28
right_ear,0.36,0.36,0.36,0.36
body_center,0.12,0.12,0.12,0.12
left_shoulder,0.01,0.01,0.01,0.01
right_shoulder,0.01,0.01,0.01,0.01
left_elbow,0.03,0.03,0.03,0.03


## 4. Trial quality flags

2 new columns in the trials table (flags only, nothing is removed):
- `wrong_person_pct`: % of the trial's frames where the camera followed someone else.
- `passes_checks`: the goal was reached **and** the trial ended normally (auto or manual) **and** `wrong_person_pct` ≤ `MAX_WRONG_PERSON_PCT` **and** at least `MIN_FRAMES_PER_TRIAL` frames.

In [12]:
trials = preprocess.add_quality_flags(trials, frames)

print(f"Trials passing all checks: {trials['passes_checks'].sum()} / {len(trials)}")
print(f"  goal not reached:                       {(~trials['goal_reached']).sum()}")
print(f"  ended failed / interrupted:             {(~trials['end_type'].isin(['auto', 'manual'])).sum()}")
print(f"  more than {MAX_WRONG_PERSON_PCT}% frames of another person: {(trials['wrong_person_pct'] > MAX_WRONG_PERSON_PCT).sum()}")
print(f"  fewer than {MIN_FRAMES_PER_TRIAL} frames:                   {(trials['n_frames'] < MIN_FRAMES_PER_TRIAL).sum()}")
print(f"First repetitions (kept, flagged): {trials['is_first_repetition'].sum()}")

pd.crosstab([trials["task_id"], trials["task_name"]], trials["passes_checks"], margins=True)

Trials passing all checks: 2695 / 2803
  goal not reached:                       84
  ended failed / interrupted:             20
  more than 10% frames of another person: 24
  fewer than 10 frames:                   26
First repetitions (kept, flagged): 188


,passes_checks,False,True,All
task_id,task_name,,,
1,Shoulder flexion with elbow extended,7,300,307
2,Shoulder abduction with elbow extended,6,272,278
3,Draw a 5 segment trajectory,10,206,216
4,Reaching different targets,21,523,544
5,Draw a circle,15,200,215
6,Horizontal reaching,12,709,721
7,Drinking test: reaching glass to mouth,32,246,278
8,Index to nose,5,239,244
All,,108,2695,2803


## Save

Writes the 3 tables to `data/processed/` as parquet files (running it again overwrites them). The sessions table has every session; trials and frames only the usable ones. `data/` is never committed (patient data).

Load them back with e.g. `pd.read_parquet(PROCESSED_ROOT / "trials.parquet")`. Frames is big: load only the columns you need with `columns=[...]`.

In [13]:
preprocess.save_tables(sessions, trials, frames)

sessions.parquet         112 rows      0.0 MB
trials.parquet         2,803 rows      0.2 MB


frames.parquet       429,731 rows    119.4 MB
